# 01. waste_by_district_year.csv 전처리

## 개요

서울시 25개 자치구 × 연도(2020~2024) 단위의 폐기물 발생·처리 통합 데이터셋을 생성합니다.

### 사용 원본 파일

| 파일명 | 추출 변수 |
|--------|----------|
| `생활계폐기물_발생량및처리현황_2020년이후_.csv` | living_waste, living_recycle, living_incineration, living_landfill |
| `쓰레기_수거현황_2020년이후_.csv` | total_waste, total_recycle, total_incineration, total_landfill |
| `지정폐기물_발생량및처리현황_2020년이후_.csv` | designated_waste |
| `건설폐기물_발생량및처리현황_2020년이후_.csv` | construction_waste |
| `사업장배출시설계폐기물_발생량및처리현황_2020년이후_.csv` | industrial_waste |
| `주민1인당_생활계폐기물배출량.csv` | per_capita_living_waste |
| `district_population.csv` (사전 생성 필요) | population, workers (household_waste, biz_nonbiz_waste 파생 계산에 사용) |

### 열 위치 메모 (원본 파일 기준)

- **생활계**: col3=발생량소계, col4=재활용소계(E열), col7=소각소계, col8=매립소계
- **쓰레기수거**: col11=처리량소계, col12=매립(M열), col13=소각(N열), col14=재활용(O열)
- **지정/건설/사업장**: col3=발생량소계
- **1인당**: col2=kg/인일

### 출력 파일
- `waste_by_district_year.csv`

## 0. 라이브러리 및 경로 설정

모든 경로는 이 노트북 파일 위치를 기준(`BASE_DIR`)으로 설정합니다.  
원본 데이터 파일들은 `DATA_DIR` 폴더에, 출력 파일은 `OUT_DIR` 폴더에 저장됩니다.

In [ ]:
import pandas as pd
import numpy as np
import os

# ── 경로 설정 ──────────────────────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))  # 노트북 위치
DATA_DIR = os.path.join(BASE_DIR, 'data')                # 원본 CSV 폴더
OUT_DIR  = os.path.join(BASE_DIR, 'output')              # 출력 폴더
os.makedirs(OUT_DIR, exist_ok=True)

GU_LIST = [
    '종로구','중구','용산구','성동구','광진구','동대문구','중랑구',
    '성북구','강북구','도봉구','노원구','은평구','서대문구','마포구',
    '양천구','강서구','구로구','금천구','영등포구','동작구','관악구',
    '서초구','강남구','송파구','강동구'
]
YEARS = [2020, 2021, 2022, 2023, 2024]

print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')
print(f'자치구 수 : {len(GU_LIST)}')
print(f'연도 범위 : {YEARS[0]} ~ {YEARS[-1]}')


## 1. 생활계폐기물 데이터 로드

### 추출 대상 열 (헤더 제외 실데이터 기준)
- **col3** : 발생량 소계 → `living_waste`
- **col4** : 재활용 소계(E열, 일반+음식물 합산) → `living_recycle`
- **col7** : 소각 소계 → `living_incineration`
- **col8** : 매립 소계 → `living_landfill`

### 필터 조건
- `구분별(2)` 열이 자치구명인 행만 추출 (소계·처리비율 행 제외)

In [ ]:
# ── 헬퍼: 문자열 → 숫자 변환 ─────────────────────────────────
def load_num(series):
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors='coerce').fillna(0)

# 생활계: col1=자치구, col2=year
# col3=발생량소계, col4=재활용소계(E열), col7=소각소계, col8=매립소계
df_raw = pd.read_csv(
    os.path.join(DATA_DIR, '생활계폐기물_발생량및처리현황_2020년이후_.csv'),
    encoding='utf-8-sig', header=None
)
df_living = df_raw[df_raw[1].isin(GU_LIST)][[1, 2, 3, 4, 7, 8]].copy()
df_living.columns = ['district', 'year', 'living_waste', 'living_recycle',
                     'living_incineration', 'living_landfill']
for c in ['living_waste', 'living_recycle', 'living_incineration', 'living_landfill']:
    df_living[c] = load_num(df_living[c])
df_living['year'] = df_living['year'].astype(int)
df_living = df_living[df_living['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_living.shape}  (기대: {len(GU_LIST)*len(YEARS)} 행)')
df_living.head()


## 2. 쓰레기수거현황 데이터 로드

### 추출 대상 열
- **col11** : 처리량 소계 → `total_waste`
- **col12** : 매립(M열) → `total_landfill`
- **col13** : 소각(N열) → `total_incineration`
- **col14** : 재활용(O열) → `total_recycle`

### 필터 조건
- `자치구별(2)` 열이 자치구명인 행만 추출 (소계 제외)

In [ ]:
# 쓰레기수거: col1=자치구, col2=year
# col11=처리량소계(L), col12=매립(M), col13=소각(N), col14=재활용(O)
df_raw2 = pd.read_csv(
    os.path.join(DATA_DIR, '쓰레기_수거현황_2020년이후_.csv'),
    encoding='utf-8-sig', header=None
)
df_total = df_raw2[df_raw2[1].isin(GU_LIST)][[1, 2, 11, 12, 13, 14]].copy()
df_total.columns = ['district', 'year', 'total_waste',
                    'total_landfill', 'total_incineration', 'total_recycle']
for c in ['total_waste', 'total_landfill', 'total_incineration', 'total_recycle']:
    df_total[c] = load_num(df_total[c])
df_total['year'] = df_total['year'].astype(int)
df_total = df_total[df_total['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_total.shape}  (기대: {len(GU_LIST)*len(YEARS)} 행)')
df_total.head()


## 3. 지정폐기물 데이터 로드

### 추출 대상 열
- **col3** : 발생량 소계 → `designated_waste`

In [ ]:
# 지정폐기물: col1=자치구, col2=year, col3=발생량소계
df_raw3 = pd.read_csv(
    os.path.join(DATA_DIR, '지정폐기물_발생량및처리현황_2020년이후_.csv'),
    encoding='utf-8-sig', header=None
)
df_desig = df_raw3[df_raw3[1].isin(GU_LIST)][[1, 2, 3]].copy()
df_desig.columns = ['district', 'year', 'designated_waste']
df_desig['designated_waste'] = load_num(df_desig['designated_waste'])
df_desig['year'] = df_desig['year'].astype(int)
df_desig = df_desig[df_desig['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_desig.shape}')
df_desig.head()


## 4. 건설폐기물 데이터 로드

### 추출 대상 열
- **col3** : 발생량 소계 → `construction_waste`

In [ ]:
# 건설폐기물: col1=자치구, col2=year, col3=발생량소계
df_raw4 = pd.read_csv(
    os.path.join(DATA_DIR, '건설폐기물_발생량및처리현황_2020년이후_.csv'),
    encoding='utf-8-sig', header=None
)
df_const = df_raw4[df_raw4[1].isin(GU_LIST)][[1, 2, 3]].copy()
df_const.columns = ['district', 'year', 'construction_waste']
df_const['construction_waste'] = load_num(df_const['construction_waste'])
df_const['year'] = df_const['year'].astype(int)
df_const = df_const[df_const['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_const.shape}')
df_const.head()


## 5. 사업장배출시설계폐기물 데이터 로드

### 추출 대상 열
- **col3** : 발생량 소계 → `industrial_waste`

In [ ]:
# 사업장배출시설계: col1=자치구, col2=year, col3=발생량소계
df_raw5 = pd.read_csv(
    os.path.join(DATA_DIR, '사업장배출시설계폐기물_발생량및처리현황_2020년이후_.csv'),
    encoding='utf-8-sig', header=None
)
df_indus = df_raw5[df_raw5[1].isin(GU_LIST)][[1, 2, 3]].copy()
df_indus.columns = ['district', 'year', 'industrial_waste']
df_indus['industrial_waste'] = load_num(df_indus['industrial_waste'])
df_indus['year'] = df_indus['year'].astype(int)
df_indus = df_indus[df_indus['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_indus.shape}')
df_indus.head()


## 6. 1인당 생활계폐기물 배출량 데이터 로드

### 추출 대상 열
- **col2** : 1인당 배출량(kg/인·일) → `per_capita_living_waste`

이 파일은 col0=자치구, col1=연도 구조입니다.

In [ ]:
# 1인당: col0=자치구, col1=year, col2=kg/인일
df_raw6 = pd.read_csv(
    os.path.join(DATA_DIR, '주민1인당_생활계폐기물배출량.csv'),
    encoding='utf-8-sig', header=None
)
df_percap = df_raw6[df_raw6[0].isin(GU_LIST)][[0, 1, 2]].copy()
df_percap.columns = ['district', 'year', 'per_capita_living_waste']
df_percap['per_capita_living_waste'] = load_num(df_percap['per_capita_living_waste'])
df_percap['year'] = df_percap['year'].astype(int)
df_percap = df_percap[df_percap['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_percap.shape}')
df_percap.head()


## 7. district_population.csv 로드

`02_district_population.ipynb`에서 생성된 파일을 로드합니다.  
이 파일의 `population`과 `workers` 컬럼으로 가정·사업장비배출 폐기물 추정에 사용합니다.

In [ ]:
# district_population.csv는 02_district_population.ipynb에서 먼저 생성해야 합니다.
df_pop = pd.read_csv(
    os.path.join(OUT_DIR, 'district_population.csv'),
    encoding='utf-8-sig'
)
print(f'shape: {df_pop.shape}')
df_pop.head()


## 8. 전체 데이터 병합

district + year 기준으로 순차적으로 left join합니다.  
기준 데이터프레임은 `df_living`(생활계)이며, 나머지는 모두 이에 합류합니다.

In [ ]:
KEY = ['district', 'year']

df = df_living.copy()
df = df.merge(df_total,  on=KEY, how='left')
df = df.merge(df_desig,  on=KEY, how='left')
df = df.merge(df_const,  on=KEY, how='left')
df = df.merge(df_indus,  on=KEY, how='left')
df = df.merge(df_percap, on=KEY, how='left')
df = df.merge(df_pop[KEY + ['population', 'workers']], on=KEY, how='left')

numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(0)

print(f'병합 결과 shape: {df.shape}')
print(f'결측값 합계: {df.isnull().sum().sum()}')
df.head()


## 9. 파생 변수 계산

### 추정 계산식

| 변수 | 계산식 | 설명 |
|------|--------|------|
| `household_waste` | `living_waste × (population / (workers + population))` | 거주 인구 비중으로 가정 폐기물 추정 |
| `biz_nonbiz_waste` | `living_waste × (workers / (workers + population))` | 종사자 비중으로 사업장비배출 폐기물 추정 |
| `recycle_rate` | `living_recycle / living_waste` | 생활계 재활용률 (0~1) |
| `incineration_rate` | `living_incineration / living_waste` | 생활계 소각률 (0~1) |
| `landfill_rate` | `living_landfill / living_waste` | 생활계 매립률 (0~1) |
| `total_recycle_rate` | `total_recycle / total_waste` | 총폐기물 재활용률 (0~1) |
| `total_incineration_rate` | `total_incineration / total_waste` | 총폐기물 소각률 (0~1) |
| `total_landfill_rate` | `total_landfill / total_waste` | 총폐기물 매립률 (0~1) |

> 분모가 0인 경우 0으로 처리하고 0~1 범위로 클리핑합니다.

In [ ]:
EPS = 1e-9

# 가정 / 사업장비배출 추정
total_person = df['workers'] + df['population'] + EPS
df['household_waste']  = (df['living_waste'] * (df['population'] / total_person)).round(1)
df['biz_nonbiz_waste'] = (df['living_waste'] * (df['workers']    / total_person)).round(1)

# 생활계 처리 비율
lw = df['living_waste'] + EPS
df['recycle_rate']      = (df['living_recycle']      / lw).clip(0, 1).round(4)
df['incineration_rate'] = (df['living_incineration'] / lw).clip(0, 1).round(4)
df['landfill_rate']     = (df['living_landfill']     / lw).clip(0, 1).round(4)

# 총폐기물 처리 비율
tw = df['total_waste'] + EPS
df['total_recycle_rate']      = (df['total_recycle']      / tw).clip(0, 1).round(4)
df['total_incineration_rate'] = (df['total_incineration'] / tw).clip(0, 1).round(4)
df['total_landfill_rate']     = (df['total_landfill']     / tw).clip(0, 1).round(4)

# 파생 계산에 쓴 population, workers는 district_population.csv에 있으므로 제거
df = df.drop(columns=['population', 'workers'])

print('파생 변수 계산 완료')
df[['district','year','household_waste','biz_nonbiz_waste',
    'recycle_rate','total_recycle_rate']].head(10)


## 10. 열 순서 정렬 및 검증

In [ ]:
COL_ORDER = [
    'district', 'year',
    'living_waste', 'living_recycle', 'living_incineration', 'living_landfill',
    'industrial_waste', 'construction_waste', 'designated_waste',
    'per_capita_living_waste',
    'total_waste', 'total_recycle', 'total_incineration', 'total_landfill',
    'household_waste', 'biz_nonbiz_waste',
    'recycle_rate', 'incineration_rate', 'landfill_rate',
    'total_recycle_rate', 'total_incineration_rate', 'total_landfill_rate',
]
df = df[COL_ORDER]

# 자치구 순서 정렬
df['district'] = pd.Categorical(df['district'], categories=GU_LIST, ordered=True)
df = df.sort_values(['district', 'year']).reset_index(drop=True)

print(f'최종 shape        : {df.shape}  (기대: {len(GU_LIST)*len(YEARS)} × {len(COL_ORDER)})')
print(f'자치구 수          : {df["district"].nunique()}')
print(f'연도 범위          : {df["year"].min()} ~ {df["year"].max()}')
print(f'결측값 합계        : {df.isnull().sum().sum()}')
print(f'생활계 재활용률 범위: {df["recycle_rate"].min():.4f} ~ {df["recycle_rate"].max():.4f}')
print(f'총폐기물 재활용률 범위: {df["total_recycle_rate"].min():.4f} ~ {df["total_recycle_rate"].max():.4f}')
df.head()


## 11. 출력 저장

In [ ]:
out_path = os.path.join(OUT_DIR, 'waste_by_district_year.csv')
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'✓ 저장 완료: {out_path}')
print(f'  파일 크기: {os.path.getsize(out_path):,} bytes')
